# Semana 2 — Calidad de datos en salud: el marco de Kahn llevado a código

**Objetivo:** auditar la capa bronze con las tres categorías del marco armonizado de calidad de datos de Kahn et al. (2016): **conformidad**, **completitud** y **plausibilidad**, y producir un reporte de calidad reutilizable.

**Productos reales usados:** `pandera` (validación declarativa de DataFrames) y pandas.

**Al terminar tendrás:** `data/reports/dq_report_s02.csv` con cada chequeo, su categoría de Kahn, el número de fallos y su estado.

**Requisito:** haber ejecutado la semana 1 (necesitas `data/bronze/`).

### Celda de configuración del entorno (se repite al inicio de cada semana)

**Explicación línea por línea**

- `import os`, `import sys`, `from pathlib import Path`: cargan tres módulos de la librería estándar. `os` lee variables de entorno, `sys` informa la versión de Python y `Path` representa rutas de archivos como objetos. **Por qué así:** `Path` evita concatenar rutas con texto (`"a" + "/" + "b"`), que falla entre sistemas operativos. **En la práctica real** todo código de datos moderno usa `pathlib`.
- `try: import google.colab` … `except ImportError`: intenta importar un módulo que solo existe en Colab. Si la importación funciona, `IN_COLAB = True`; si lanza `ImportError`, estamos en otra máquina. **Por qué importa:** el mismo notebook corre en Colab y en tu laptop sin editarlo. **Cuándo se usa:** siempre que un código deba ser portable entre entornos.
- `from google.colab import drive` y `drive.mount("/content/drive")`: conectan tu Google Drive al sistema de archivos de Colab. **Por qué:** el disco de Colab se borra al cerrar la sesión; en Drive los datos de la semana 1 siguen ahí el lunes siguiente. La primera vez Colab te pedirá autorizar el acceso.
- `PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")`: carpeta raíz del proyecto dentro de tu Drive.
- `Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()`: fuera de Colab usa la variable de entorno `HEALTHETL_PROJECT_DIR` si existe; si no, la carpeta actual (`"."`). `.resolve()` la convierte en ruta absoluta. **En la práctica real** la configuración se pasa por variables de entorno, nunca escrita a mano dentro del código (principio *12-factor app*).
- `DATA_DIR = PROJECT_DIR / "data"`: el operador `/` de `pathlib` une rutas de forma segura.
- `DATA_DIR.mkdir(parents=True, exist_ok=True)`: crea la carpeta y sus padres; `exist_ok=True` evita un error si ya existe. Esto hace la celda **idempotente**: puedes ejecutarla 1 o 10 veces con el mismo resultado, una propiedad central de todo pipeline ETL.
- Los tres `print`: confirman versión de Python, entorno y ruta. Registrar el entorno al inicio es un hábito de reproducibilidad.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # este módulo solo existe dentro de Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")
else:
    PROJECT_DIR = Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("¿Colab?:", IN_COLAB)
print("Carpeta de datos:", DATA_DIR)

### Paso 1 — Instalar y cargar librerías

**Explicación línea por línea**

- `%pip install -q "pandera>=0.24"`: `%pip` es un comando mágico de Jupyter/Colab que instala en el mismo entorno del notebook (a diferencia de `!pip`, que puede apuntar a otro Python). `-q` reduce el ruido. La restricción `>=0.24` asegura la API moderna `pandera.pandas`.
- `import pandas as pd`: tablas en memoria.
- `import pandera`: el paquete raíz, solo para imprimir su versión al final (registrar versiones es parte de la reproducibilidad).
- `import pandera.pandas as pa`: importa el *backend* de pandas de pandera. **Por qué pandera:** permite declarar reglas ("esta columna es un UUID", "este valor está en esta lista") como un esquema versionable, en lugar de esconderlas en `if` sueltos. **En la práctica real** equipos de datos usan herramientas equivalentes (pandera, Great Expectations, las *expectations* de Databricks) para que las reglas de calidad sean código revisable.
- `BRONZE_DIR`, `REPORTS_DIR`: rutas de entrada y salida; `mkdir(..., exist_ok=True)` es idempotente.
- `assert (BRONZE_DIR / "_manifest.json").exists()`: falla temprano con un mensaje claro si no corriste la semana 1.

In [ ]:
%pip install -q "pandera>=0.24"

In [ ]:
import pandas as pd
import pandera
import pandera.pandas as pa

BRONZE_DIR = DATA_DIR / "bronze"
REPORTS_DIR = DATA_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
assert (BRONZE_DIR / "_manifest.json").exists(), "Ejecuta primero la semana 1"
print("pandas", pd.__version__, "| pandera", pandera.__version__)

### Paso 2 — Leer bronze y preparar el registro de chequeos

**Explicación línea por línea**

- `pd.read_parquet(BRONZE_DIR / "patients.parquet")`: lee la tabla completa; en bronze todas las columnas son texto.
- Para `observations` pasamos `columns=[...]` y leemos solo las columnas necesarias: con 531 mil filas, ahorrar columnas ahorra memoria.
- `dq_results = []`: lista donde cada chequeo dejará una fila.
- `def log_check(...)`: función que estandariza cómo se registra un chequeo: identificador, categoría y subcategoría de Kahn, tabla, fallos, total, descripción.
- `pct = round(100 * n_fail / n_total, 3) if n_total else 0.0`: porcentaje de fallos; la condición evita dividir entre cero.
- `"status": "PASS" if n_fail == 0 else "FAIL"`: estado binario. **En la práctica real** se agregan umbrales (por ejemplo, aceptar hasta 5 % de fallos); el *Data Quality Dashboard* de OHDSI define un umbral por chequeo.
- `return n_fail`: devolver el valor permite usarlo en un `assert` más adelante.

In [ ]:
patients = pd.read_parquet(BRONZE_DIR / "patients.parquet")
encounters = pd.read_parquet(BRONZE_DIR / "encounters.parquet")
conditions = pd.read_parquet(BRONZE_DIR / "conditions.parquet")
observations = pd.read_parquet(
    BRONZE_DIR / "observations.parquet",
    columns=["DATE", "PATIENT", "ENCOUNTER", "CATEGORY", "CODE", "DESCRIPTION", "VALUE", "UNITS", "TYPE"],
)

dq_results = []


def log_check(check_id, category, subcategory, table, n_fail, n_total, description):
    pct = round(100 * n_fail / n_total, 3) if n_total else 0.0
    dq_results.append({
        "check_id": check_id, "kahn_category": category, "kahn_subcategory": subcategory,
        "table": table, "n_fail": int(n_fail), "n_total": int(n_total), "pct_fail": pct,
        "status": "PASS" if n_fail == 0 else "FAIL", "description": description,
    })
    return n_fail


print(len(patients), len(encounters), len(conditions), len(observations))

### Paso 3 — Conformidad de valores con esquemas pandera

Kahn define la **conformidad de valores** como la concordancia de los datos con restricciones de formato predefinidas. Aquí validamos el texto crudo de bronze contra patrones (expresiones regulares) y listas de valores permitidos.

**Explicación línea por línea**

- `UUID = r"^[0-9a-f]{8}-..."`: expresión regular de un identificador UUID en minúsculas. La `r` delante evita que Python interprete `\` como escape. `^` y `$` anclan el patrón al inicio y fin del texto.
- `DATE`, `DATE_OR_EMPTY`, `DATETIME`: fecha `AAAA-MM-DD`, fecha o vacío (`(...)?` hace opcional el grupo), y fecha-hora ISO 8601 en UTC (`Z` al final).
- `ENCOUNTER_CLASSES = [...]`: dominio permitido de `ENCOUNTERCLASS` observado en la semana 1.
- `pa.DataFrameSchema({...})`: declara las reglas por columna.
- `pa.Column(str, pa.Check.str_matches(UUID), unique=True)`: la columna debe ser texto, cumplir el patrón y no repetirse. `unique=True` es un chequeo de **plausibilidad de unicidad** en Kahn.
- `pa.Check.isin(["M", "F"])`: valor dentro de un conjunto cerrado.
- `pa.Check.str_length(min_value=1)`: texto no vacío.
- `def run_schema(schema, df):` ejecuta la validación. `lazy=True` hace que pandera recorra **todas** las reglas y junte todos los fallos, en vez de detenerse en el primero. **Por qué importa:** un reporte de calidad necesita la lista completa.
- `except pa.errors.SchemaErrors as exc: return exc.failure_cases`: `failure_cases` es un DataFrame con columna, chequeo y valor que falló.

In [ ]:
UUID = r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$"
DATE = r"^\d{4}-\d{2}-\d{2}$"
DATE_OR_EMPTY = r"^(\d{4}-\d{2}-\d{2})?$"
DATETIME = r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$"
ENCOUNTER_CLASSES = ["wellness", "ambulatory", "outpatient", "urgentcare", "emergency", "inpatient"]

patients_schema = pa.DataFrameSchema({
    "Id": pa.Column(str, pa.Check.str_matches(UUID), unique=True),
    "BIRTHDATE": pa.Column(str, pa.Check.str_matches(DATE)),
    "DEATHDATE": pa.Column(str, pa.Check.str_matches(DATE_OR_EMPTY)),
    "GENDER": pa.Column(str, pa.Check.isin(["M", "F"])),
})
encounters_schema = pa.DataFrameSchema({
    "Id": pa.Column(str, pa.Check.str_matches(UUID), unique=True),
    "PATIENT": pa.Column(str, pa.Check.str_matches(UUID)),
    "START": pa.Column(str, pa.Check.str_matches(DATETIME)),
    "STOP": pa.Column(str, pa.Check.str_matches(DATETIME)),
    "ENCOUNTERCLASS": pa.Column(str, pa.Check.isin(ENCOUNTER_CLASSES)),
})
observations_schema = pa.DataFrameSchema({
    "PATIENT": pa.Column(str, pa.Check.str_matches(UUID)),
    "DATE": pa.Column(str, pa.Check.str_matches(DATETIME)),
    "CODE": pa.Column(str, pa.Check.str_length(min_value=1)),
    "TYPE": pa.Column(str, pa.Check.isin(["numeric", "text"])),
})


def run_schema(schema, df):
    try:
        schema.validate(df, lazy=True)
        return pd.DataFrame(columns=["column", "check", "failure_case"])
    except pa.errors.SchemaErrors as exc:
        return exc.failure_cases

### Paso 4 — Ejecutar los esquemas y registrar resultados por regla

**Explicación línea por línea**

- `def rule_failures(schema, df):` devuelve una tabla con una fila por (columna, regla) y su número de fallos.
- `fallos = run_schema(schema, df)`: obtiene todos los casos fallidos de una vez.
- `for col_name, column in schema.columns.items():` recorre las columnas del esquema.
- `reglas = [(c.name, c.error) for c in column.checks]`: cada chequeo de pandera tiene un nombre corto (`isin`) y una firma (`isin(['M', 'F'])`); la firma es exactamente el texto que pandera escribe en la columna `check` de `failure_cases`, por eso se usa para emparejar.
- `if column.unique: reglas.append(("field_uniqueness", "field_uniqueness"))`: la unicidad no es un `Check` sino una propiedad de la columna; pandera la reporta con ese nombre.
- `n = int(((fallos["column"] == col_name) & (fallos["check"].astype(str) == firma)).sum())`: cuenta los fallos de esa columna y regla. `.astype(str)` asegura comparar texto con texto.
- `def log_schema(table, schema, df):` toma esa tabla y registra cada regla con `log_check`. Unicidad va a Plausibilidad; el resto a Conformidad de valores.
- Al final mostramos los chequeos de esta etapa con `pd.DataFrame(dq_results)`.

In [ ]:
def rule_failures(schema, df):
    fallos = run_schema(schema, df)
    filas = []
    for col_name, column in schema.columns.items():
        reglas = [(c.name, c.error) for c in column.checks]
        if column.unique:
            reglas.append(("field_uniqueness", "field_uniqueness"))
        for nombre, firma in reglas:
            n = int(((fallos["column"] == col_name) & (fallos["check"].astype(str) == firma)).sum())
            filas.append({"column": col_name, "rule": nombre, "n_fail": n})
    return pd.DataFrame(filas)


def log_schema(table, schema, df):
    for r in rule_failures(schema, df).itertuples():
        is_unique = r.rule == "field_uniqueness"
        log_check(
            check_id=f"{table}.{r.column}.{r.rule}",
            category="Plausibility" if is_unique else "Conformance",
            subcategory="Uniqueness" if is_unique else "Value",
            table=table, n_fail=r.n_fail, n_total=len(df),
            description=f"{r.column} cumple {r.rule}",
        )


log_schema("patients", patients_schema, patients)
log_schema("encounters", encounters_schema, encounters)
log_schema("observations", observations_schema, observations)
pd.DataFrame(dq_results)[["check_id", "kahn_category", "n_fail", "status"]]

### Paso 5 — Conversión de tipos auditada (texto → fecha y número)

**Explicación línea por línea**

- `def parse_dt(s):` convierte texto a fecha-hora.
- `s.where(s != "")`: reemplaza los textos vacíos por `NaN` (valor ausente) antes de convertir; así un vacío no se cuenta como error de formato.
- `pd.to_datetime(..., format="ISO8601", utc=True, errors="coerce")`: interpreta el estándar ISO 8601, deja todo en UTC y, si algo no se puede convertir, lo vuelve `NaT` (fecha ausente) en vez de detener el programa.
- `def parse_num(s):` igual para números con `pd.to_numeric(..., errors="coerce")`.
- `def coercion_failures(raw, parsed):` cuenta valores que **no estaban vacíos** pero quedaron ausentes tras convertir: esos son fallos de conformidad que de otro modo pasarían inadvertidos. **En la práctica real** `errors="coerce"` sin este conteo es una de las fuentes más comunes de pérdida silenciosa de datos.
- `obs_num = observations["TYPE"] == "numeric"`: solo las observaciones declaradas numéricas deben convertirse a número (las de tipo `text` son respuestas como "Never smoker").

In [ ]:
def parse_dt(s):
    return pd.to_datetime(s.where(s != ""), format="ISO8601", utc=True, errors="coerce")


def parse_num(s):
    return pd.to_numeric(s.where(s != ""), errors="coerce")


def coercion_failures(raw, parsed):
    return int(((raw != "") & parsed.isna()).sum())


pt = patients.assign(BIRTHDATE=parse_dt(patients["BIRTHDATE"]), DEATHDATE=parse_dt(patients["DEATHDATE"]))
en = encounters.assign(START=parse_dt(encounters["START"]), STOP=parse_dt(encounters["STOP"]))
obs_num = observations["TYPE"] == "numeric"
ob = observations.assign(DATE=parse_dt(observations["DATE"]), VALUE_NUM=parse_num(observations["VALUE"].where(obs_num, "")))

log_check("patients.BIRTHDATE.parse", "Conformance", "Value", "patients",
          coercion_failures(patients["BIRTHDATE"], pt["BIRTHDATE"]), len(pt), "BIRTHDATE convertible a fecha")
log_check("encounters.START.parse", "Conformance", "Value", "encounters",
          coercion_failures(encounters["START"], en["START"]), len(en), "START convertible a fecha-hora")
log_check("observations.VALUE.numeric_parse", "Conformance", "Value", "observations",
          coercion_failures(observations["VALUE"].where(obs_num, ""), ob["VALUE_NUM"]), int(obs_num.sum()),
          "VALUE numérico cuando TYPE='numeric'")
ob[["DATE", "CODE", "VALUE", "VALUE_NUM", "UNITS"]].head()

### Paso 6 — Conformidad relacional (claves foráneas)

Kahn define la **conformidad relacional** como la concordancia con restricciones estructurales de la base de datos, como claves primarias y foráneas.

**Explicación línea por línea**

- `def fk_orphans(child, child_col, parent, parent_col):` cuenta filas "huérfanas": el valor de la tabla hija no existe en la tabla padre.
- `~child[child_col].isin(parent[parent_col])`: `isin` devuelve verdadero si el valor está en el padre; `~` lo niega.
- `child[child_col] != ""`: no contamos como huérfano un campo vacío (eso es completitud, no relación).
- Revisamos cuatro relaciones: encuentro→paciente, condición→paciente, condición→encuentro y observación→encuentro. **Por qué importa:** un encuentro sin paciente rompe cualquier análisis por persona; en datos reales aparece cuando se carga una tabla antes que otra o se purgan pacientes sin propagar el borrado.

In [ ]:
def fk_orphans(child, child_col, parent, parent_col):
    return int(((child[child_col] != "") & ~child[child_col].isin(parent[parent_col])).sum())


for child_name, child, col, parent_name, parent, pcol in [
    ("encounters", encounters, "PATIENT", "patients", patients, "Id"),
    ("conditions", conditions, "PATIENT", "patients", patients, "Id"),
    ("conditions", conditions, "ENCOUNTER", "encounters", encounters, "Id"),
    ("observations", observations, "ENCOUNTER", "encounters", encounters, "Id"),
]:
    n = fk_orphans(child, col, parent, pcol)
    log_check(f"{child_name}.{col}->{parent_name}.{pcol}", "Conformance", "Relational",
              child_name, n, len(child), f"{col} existe en {parent_name}.{pcol}")
    print(f"{child_name}.{col} → {parent_name}: {n} huérfanos")

### Paso 7 — Conformidad computacional: ¿el IMC registrado coincide con peso/talla²?

Kahn incluye la **conformidad computacional**: los valores derivados deben coincidir con su fórmula. El índice de masa corporal (LOINC 39156-5) se calcula como peso (kg, LOINC 29463-7) dividido por talla (m, LOINC 8302-2) al cuadrado.

**Explicación línea por línea**

- `vit = ob[ob["CODE"].isin([...])]`: filtra las tres mediciones.
- `.pivot_table(index=["ENCOUNTER"], columns="CODE", values="VALUE_NUM", aggfunc="last")`: convierte filas en columnas: una fila por encuentro, una columna por código. `aggfunc="last"` resuelve si hubiera dos valores del mismo código en el encuentro.
- `.dropna()`: conserva solo encuentros con las tres mediciones.
- `bmi_calc = w["29463-7"] / (w["8302-2"] / 100) ** 2`: la talla viene en centímetros; se divide entre 100 para pasar a metros.
- `diff = (bmi_calc - w["39156-5"]).abs()`: diferencia absoluta.
- `n_fail = int((diff > 0.5).sum())`: consideramos fallo una diferencia mayor a 0,5 kg/m² (tolerancia elegida para este ejercicio, por redondeos).

In [ ]:
vit = ob[ob["CODE"].isin(["29463-7", "8302-2", "39156-5"])]
w = vit.pivot_table(index="ENCOUNTER", columns="CODE", values="VALUE_NUM", aggfunc="last").dropna()
bmi_calc = w["29463-7"] / (w["8302-2"] / 100) ** 2
diff = (bmi_calc - w["39156-5"]).abs()
n_fail = int((diff > 0.5).sum())
log_check("observations.BMI.computed", "Conformance", "Computational", "observations",
          n_fail, len(w), "IMC registrado = peso / talla² (±0,5)")
print(f"Encuentros con peso, talla e IMC: {len(w):,}; discrepancias > 0,5: {n_fail}")
diff.describe()

### Paso 8 — Completitud atemporal y temporal

Kahn distingue la **completitud atemporal** (presencia de datos en un momento) de la **temporal** (presencia a lo largo del tiempo).

**Explicación línea por línea**

- `(df == "").mean() * 100`: en bronze los ausentes son texto vacío; comparar con `""` da verdadero/falso y `.mean()` da la proporción.
- `.round(1).sort_values(ascending=False).head(8)`: muestra las columnas más incompletas. Que `DEATHDATE` esté vacía en la mayoría es **esperado** (la mayoría está viva): completitud baja no siempre es un error; se interpreta según el significado clínico.
- `n_sin_enc = (~pt["Id"].isin(en["PATIENT"])).sum()`: pacientes sin ningún encuentro (un paciente sin contacto con el sistema no aporta datos clínicos).
- `por_anio = en["START"].dt.year.value_counts().sort_index()`: encuentros por año. `.dt.year` extrae el año.
- `ultimos = por_anio.loc[2005:2021]`: últimos años completos. Un año con caída brusca suele indicar un problema de extracción (por ejemplo, un cambio de sistema en un hospital), no un cambio real en la salud de la población.

In [ ]:
for name, df in [("patients", patients), ("encounters", encounters), ("conditions", conditions)]:
    print(f"\n% vacío por columna en {name}:")
    print(((df == "").mean() * 100).round(1).sort_values(ascending=False).head(8).to_string())

n_sin_enc = int((~pt["Id"].isin(en["PATIENT"])).sum())
log_check("patients.has_encounter", "Completeness", "Atemporal", "patients", n_sin_enc, len(pt),
          "Todo paciente tiene al menos un encuentro")

por_anio = en["START"].dt.year.value_counts().sort_index()
ultimos = por_anio.loc[2005:2021]
print("\nEncuentros por año 2005-2021:")
print(ultimos.to_string())

### Paso 9 — Plausibilidad atemporal: rangos fisiológicos

**Explicación línea por línea**

- `PLAUSIBLE = {...}`: diccionario `código LOINC → (mínimo, máximo, nombre)`. Los límites son amplios a propósito: no buscan valores *anormales* (eso es clínica) sino valores *imposibles* (eso es calidad de datos). Son límites elegidos para este ejercicio; en un proyecto real se documentan y los valida un clínico.
- `for code, (lo, hi, label) in PLAUSIBLE.items():` recorre cada código desempaquetando la tupla.
- `vals = ob.loc[ob["CODE"] == code, "VALUE_NUM"].dropna()`: valores de ese código.
- `n_out = int(((vals < lo) | (vals > hi)).sum())`: `|` es "o" elemento a elemento.
- `log_check(...)`: registra el resultado como Plausibilidad/Atemporal.

In [ ]:
PLAUSIBLE = {
    "8480-6": (50, 260, "Presión sistólica (mm[Hg])"),
    "8462-4": (20, 160, "Presión diastólica (mm[Hg])"),
    "39156-5": (10, 80, "IMC (kg/m2)"),
    "8867-4": (20, 250, "Frecuencia cardiaca (/min)"),
    "8302-2": (30, 230, "Talla (cm)"),
    "29463-7": (0.5, 350, "Peso (kg)"),
}
for code, (lo, hi, label) in PLAUSIBLE.items():
    vals = ob.loc[ob["CODE"] == code, "VALUE_NUM"].dropna()
    n_out = int(((vals < lo) | (vals > hi)).sum())
    log_check(f"observations.{code}.range", "Plausibility", "Atemporal", "observations",
              n_out, len(vals), f"{label} en [{lo}, {hi}]")
    print(f"{label:<28} n={len(vals):>6,}  fuera de rango={n_out}")

### Paso 10 — Plausibilidad temporal: el orden de los eventos debe tener sentido

**Explicación línea por línea**

- `death_lt_birth`: muertes anteriores al nacimiento; `notna()` evita comparar con fechas ausentes.
- `stop_lt_start`: encuentros que terminan antes de empezar.
- `e2 = en.merge(pt[["Id", "BIRTHDATE", "DEATHDATE"]], left_on="PATIENT", right_on="Id", suffixes=("", "_pt"))`: une cada encuentro con las fechas del paciente. `suffixes` evita que la columna `Id` del paciente pise la del encuentro.
- `before_birth`: encuentros antes de nacer (tolerancia: mismo día).
- `after_death`: encuentros que empiezan más de un día después de la muerte. `pd.Timedelta(days=1)` representa una duración. **Por qué tolerancia:** las fechas de muerte suelen registrarse sin hora; sin tolerancia, un evento del mismo día parecería imposible.

In [ ]:
death_lt_birth = int((pt["DEATHDATE"].notna() & (pt["DEATHDATE"] < pt["BIRTHDATE"])).sum())
stop_lt_start = int((en["STOP"] < en["START"]).sum())
e2 = en.merge(pt[["Id", "BIRTHDATE", "DEATHDATE"]], left_on="PATIENT", right_on="Id", suffixes=("", "_pt"))
before_birth = int((e2["START"] < e2["BIRTHDATE"] - pd.Timedelta(days=1)).sum())
after_death = int((e2["DEATHDATE"].notna() & (e2["START"] > e2["DEATHDATE"] + pd.Timedelta(days=1))).sum())

log_check("patients.death_after_birth", "Plausibility", "Temporal", "patients", death_lt_birth, len(pt), "DEATHDATE ≥ BIRTHDATE")
log_check("encounters.stop_after_start", "Plausibility", "Temporal", "encounters", stop_lt_start, len(en), "STOP ≥ START")
log_check("encounters.after_birth", "Plausibility", "Temporal", "encounters", before_birth, len(e2), "Encuentro después de nacer")
log_check("encounters.before_death", "Plausibility", "Temporal", "encounters", after_death, len(e2), "Encuentro antes de morir (+1 día)")
print(death_lt_birth, stop_lt_start, before_birth, after_death)

### Paso 11 — Prueba de los chequeos: inyectar errores realistas en una copia

El FAQ de Synthea advierte que sus registros no tienen datos faltantes ni incorrectos, a diferencia de los datos reales. Un chequeo que nunca falla no demuestra nada: hay que comprobar que **detecta** errores. Esto se llama probar con datos "sembrados" (*fault injection*).

**Explicación línea por línea**

- `dirty = patients.copy()`: copia independiente; nunca ensuciamos el original.
- `dirty.loc[dirty.index[0], "GENDER"] = "male"`: valor fuera del dominio (error típico al integrar un sistema que codifica distinto).
- `dirty.loc[dirty.index[1], "BIRTHDATE"] = "11/05/1998"`: formato de fecha local (día/mes/año), muy común en hojas de cálculo australianas o latinoamericanas.
- `pd.concat([dirty, dirty.iloc[[2]]])`: duplica una fila para romper la unicidad del `Id`.
- `ignore_index=True`: renumera el índice tras concatenar.
- `rf = rule_failures(patients_schema, dirty)`: vuelve a correr **la misma función** que alimenta el reporte. Así pruebas el camino real, no una copia.
- `rf[rf["n_fail"] > 0]`: debes ver 1 fallo de `GENDER`, 1 de `BIRTHDATE` y 2 de unicidad de `Id` (pandera reporta ambas filas repetidas).
- `assert rf["n_fail"].sum() == 4`: el notebook se detiene si los chequeos no detectan exactamente lo sembrado. **En la práctica real** esto es una prueba unitaria de tus reglas de calidad; en la semana 8 la convertirás en un test de `pytest`.

In [ ]:
dirty = patients.copy()
dirty.loc[dirty.index[0], "GENDER"] = "male"
dirty.loc[dirty.index[1], "BIRTHDATE"] = "11/05/1998"
dirty = pd.concat([dirty, dirty.iloc[[2]]], ignore_index=True)

rf = rule_failures(patients_schema, dirty)
print(rf[rf["n_fail"] > 0])
assert rf["n_fail"].sum() == 4, "Los chequeos no detectaron exactamente los errores sembrados"

### Paso 12 — Investigar un fallo antes de "corregirlo"

El chequeo `encounters.before_death` falla. Antes de borrar filas hay que entender **qué** son.

**Explicación línea por línea**

- `tardios = e2[...]`: reutiliza la misma condición del paso 10 para quedarse con los encuentros posteriores a la muerte.
- `tardios["DESCRIPTION"].value_counts().head()`: qué tipo de encuentros son. Verás que la gran mayoría son `Death Certification` (certificación de defunción).
- `(tardios["START"] - tardios["DEATHDATE"]).dt.days.describe()`: cuántos días después de la muerte ocurren. Restar dos fechas da una duración; `.dt.days` la expresa en días.
- `clinicos = tardios[tardios["DESCRIPTION"] != "Death Certification"]`: excluye el proceso administrativo, que es **plausible** después de morir.
- `log_check("encounters.before_death_clinical", ...)`: registra la regla refinada. **Lección de práctica real:** la plausibilidad depende del contexto clínico y del proceso asistencial; una regla se refina con conocimiento del dominio y la excepción se documenta, no se esconde. Los encuentros clínicos restantes son anomalías del generador que quedan documentadas como limitación.

In [ ]:
tardios = e2[e2["DEATHDATE"].notna() & (e2["START"] > e2["DEATHDATE"] + pd.Timedelta(days=1))]
print(tardios["DESCRIPTION"].value_counts().head())
print((tardios["START"] - tardios["DEATHDATE"]).dt.days.describe())

clinicos = tardios[tardios["DESCRIPTION"] != "Death Certification"]
log_check("encounters.before_death_clinical", "Plausibility", "Temporal", "encounters", len(clinicos), len(e2),
          "Encuentro clínico antes de morir (excluye certificación de defunción)")
print("Encuentros clínicos después de la muerte:", len(clinicos))

### Paso 13 — Reporte de calidad

**Explicación línea por línea**

- `report = pd.DataFrame(dq_results)`: convierte la lista de diccionarios en tabla.
- `report.to_csv(REPORTS_DIR / "dq_report_s02.csv", index=False)`: guarda el reporte; CSV porque lo abrirá también gente no técnica.
- `report.groupby(["kahn_category", "status"]).size().unstack(fill_value=0)`: resumen por categoría de Kahn: cuántos chequeos pasaron y fallaron. `unstack` pasa `status` a columnas; `fill_value=0` rellena combinaciones inexistentes.
- `report[report["status"] == "FAIL"]`: lista los chequeos que fallaron para investigarlos. **En la práctica real** cada fallo termina en una de tres acciones: corregir en origen, corregir en la transformación documentando la regla, o aceptar y documentar la limitación.

In [ ]:
report = pd.DataFrame(dq_results)
report.to_csv(REPORTS_DIR / "dq_report_s02.csv", index=False)
print(report.groupby(["kahn_category", "status"]).size().unstack(fill_value=0))
report[report["status"] == "FAIL"][["check_id", "n_fail", "n_total", "pct_fail", "description"]]

### Autoevaluación

1. Clasifica en Kahn: (a) `GENDER = "male"`, (b) encuentro sin paciente, (c) IMC que no cuadra con peso y talla, (d) diastólica de 400.
2. ¿Por qué `errors="coerce"` sin conteo de fallos es peligroso?
3. `DEATHDATE` está vacía en la mayoría de pacientes. ¿Es un problema de completitud? Justifica.
4. ¿Por qué sembrar errores es necesario cuando trabajas con Synthea?

**Commit sugerido:** `feat(s02): reporte de calidad de datos según marco de Kahn con pandera`